# Chunking strategies showdown: fixed, semantic, hierarchical

Three engineers on the customer-support RAG team have been arguing in PRs for two weeks. One wants 512-token fixed chunks. One wants semantic chunks split on sentence boundaries. One wants hierarchical chunks split by markdown section then paragraph. The eval lead is tired of opinions and wants numbers: index all three against the same corpus, run the same 50 queries, report recall@5, mean retrieval latency, and storage. The deliverable is a defended recommendation against a 50ms p95 latency budget.

You have 90 minutes to:
1. Generate a synthetic 2 MB markdown corpus with three header levels.
2. Implement three chunkers and run the corpus through each.
3. Embed every chunk with `text-embedding-3-small` and write to three Supabase pgvector tables.
4. Run a 50-query eval against each table and write a comparison JSON to `/content/eval_results.json`.
5. Defend a production choice in the reflection cell.

**Time.** Plan for about 90 minutes hands-on. If the eval loop crashes mid-run or you re-tune the chunkers, budget up to 120 minutes. The cleanup cell at the bottom drops all three tables so your Supabase project storage returns to baseline.

**Cost.** This lab costs about a dollar if everything goes perfectly on the first try. It will not go perfectly on the first try (that is the point of doing labs). A realistic session with two or three eval re-runs lands around two dollars. The OpenAI embedding API is the main spend; the Supabase free tier covers the vector storage; the eval queries are pennies. The cleanup cell drops all three tables so your project stops storing the test data the moment you finish.

This lab maps to the AI/ML Engineering track Sub-track A (retrieval systems) and is the gateway lab for the rest of the sub-track.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the lab's runtime dependencies. All versions
# pinned per LAB-CREATION-BLUEPRINT.md build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0 openai==1.51.0 supabase==2.7.4 psycopg2-binary==2.9.9 numpy==1.26.4 tiktoken==0.7.0 requests==2.32.3

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Table names use Postgres-valid identifiers
# (underscores, not hyphens) per the brief. The labrun:lab-slug tag value
# stays hyphenated for orphan-scan parity with AWS labs.

import atexit
import getpass
import json
import os
import random
import re
import sys
import time
from typing import Iterable

import numpy as np
import tiktoken

from labrun_checks import (
    CheckpointResult,
    CleanupResource,
    check,
    cleanup,
    register,
    run_cleanup,
)

LAB_ID = "chunking-strategies-comparison"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # hyphenated tag value, matches the cert YAML slug

# Postgres tables: underscores only. The shared prefix is what the orphan scan
# and verification scan both filter by.
TABLE_PREFIX = "labrun_chunking_strategies_comparison_"
STRATEGIES = ["fixed", "semantic", "hierarchical"]
TABLE_NAMES = {s: f"{TABLE_PREFIX}{s}" for s in STRATEGIES}

EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM = 1536  # text-embedding-3-small is 1536-dim. Mismatch breaks vector(1536).

# Deterministic seed for the synthetic corpus and the eval label set so the
# numbers are repeatable across re-runs.
CORPUS_SEED = 42
CORPUS_PAGE_COUNT = 150
EVAL_QUERY_COUNT = 50

# Fixed-chunk parameters per brief.
FIXED_CHUNK_TOKENS = 512
FIXED_CHUNK_OVERLAP = 64
FIXED_CHUNK_STRIDE = FIXED_CHUNK_TOKENS - FIXED_CHUNK_OVERLAP  # 448 tokens per window advance

EVAL_RESULTS_PATH = "/content/eval_results.json"

print(f"Lab id: {LAB_ID}")
print(f"Tables to be created: {list(TABLE_NAMES.values())}")

In [ ]:
# NBVAL_SKIP
# Register the session, prompt for credentials, validate each one, enable
# the pgvector extension. This cell must succeed before the manifest cell
# creates anything.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY (sk-...): ")
SUPABASE_URL = getpass.getpass("SUPABASE_URL (https://<project>.supabase.co): ").strip()
SUPABASE_SERVICE_ROLE_KEY = getpass.getpass("SUPABASE_SERVICE_ROLE_KEY: ")
BEDROCK_ACCESS_KEY_ID = getpass.getpass(
    "BEDROCK_ACCESS_KEY_ID (optional, leave blank to skip the bonus cell): "
).strip()
BEDROCK_SECRET_ACCESS_KEY = getpass.getpass(
    "BEDROCK_SECRET_ACCESS_KEY (optional, leave blank to skip the bonus cell): "
).strip()

# Validate OpenAI immediately. A 401 here fails fast rather than during the
# first embed call halfway through the lab.
from openai import OpenAI, AuthenticationError

openai_client = OpenAI(api_key=OPENAI_API_KEY)
try:
    openai_client.models.list()
except AuthenticationError as e:
    print("OpenAI credentials rejected. Re-run this cell with a fresh key.")
    print(f"  {e}")
    raise SystemExit(1)
print("OpenAI credentials validated.")

# Validate Supabase via a service-role auth ping. The simplest auth-bearing
# call is a small SQL via the REST API; we use psycopg2 because the rest of
# the lab also writes vectors via SQL.
import psycopg2
from urllib.parse import urlparse

# Supabase project URL looks like https://<project>.supabase.co. The Postgres
# host is derived from the project ref. Default Postgres port is 5432.
parsed = urlparse(SUPABASE_URL)
project_ref = parsed.hostname.split(".")[0] if parsed.hostname else ""
if not project_ref:
    print("SUPABASE_URL must be the project URL, e.g. https://<project-ref>.supabase.co.")
    raise SystemExit(1)

PG_HOST = f"db.{project_ref}.supabase.co"
PG_PORT = 5432
PG_DBNAME = "postgres"
PG_USER = "postgres"
PG_PASSWORD = SUPABASE_SERVICE_ROLE_KEY  # service role doubles as the Postgres password for the default db

def pg_connect():
    return psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DBNAME,
        user=PG_USER,
        password=PG_PASSWORD,
        sslmode="require",
        connect_timeout=10,
    )

try:
    with pg_connect() as conn:
        with conn.cursor() as cur:
            cur.execute("select version()")
            version = cur.fetchone()[0]
except Exception as e:
    print("Supabase Postgres connection failed. Check SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY.")
    print(f"  {e}")
    raise SystemExit(1)
print(f"Supabase Postgres connected: {version.split(',')[0]}")

# Enable pgvector. Brief calls this out as a top failure mode if the student
# forgets to enable the extension before creating vector(1536) columns.
with pg_connect() as conn:
    with conn.cursor() as cur:
        cur.execute("create extension if not exists vector")
    conn.commit()
print("pgvector extension enabled.")

# Bedrock keys are optional. The bonus cell skips itself if either is blank.
BEDROCK_AVAILABLE = bool(BEDROCK_ACCESS_KEY_ID and BEDROCK_SECRET_ACCESS_KEY)
print(f"Bedrock bonus cell: {'available' if BEDROCK_AVAILABLE else 'skipped (keys blank)'}")

# Credentials dict for the cleanup adapter. Keys are never written to disk.
LAB_CREDENTIALS = {
    "supabase_url": SUPABASE_URL,
    "supabase_service_role_key": SUPABASE_SERVICE_ROLE_KEY,
    "pg_host": PG_HOST,
    "pg_port": PG_PORT,
    "pg_user": PG_USER,
    "pg_password": PG_PASSWORD,
    "pg_dbname": PG_DBNAME,
}

register(session_token=session_token, lab_id=LAB_ID, credentials=LAB_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, custom SupabaseCleanupAdapter, atexit safety net, orphan
# scan. The adapter implements the labrun-checks CleanupAdapter protocol
# (delete_resource, describe_resource, tag_scan). When the labrun-checks
# vector_db.py adapter ships, the in-notebook adapter below can be retired.
# TODO: migrate to vector_db.py adapter once shipped.

from labrun_checks import CleanupAdapter as _CleanupAdapterProtocol  # imported for type clarity only


class SupabaseCleanupAdapter:
    """Drops Supabase pgvector tables for this lab.

    Implements the labrun-checks CleanupAdapter protocol so run_cleanup
    can orchestrate the three-phase teardown (delete, verify, tag audit).
    """

    def _connect(self, credentials: dict):
        return psycopg2.connect(
            host=credentials["pg_host"],
            port=credentials["pg_port"],
            dbname=credentials["pg_dbname"],
            user=credentials["pg_user"],
            password=credentials["pg_password"],
            sslmode="require",
            connect_timeout=10,
        )

    def delete_resource(self, credentials: dict, resource: CleanupResource) -> None:
        if resource.type == "supabase_pgvector_table":
            with self._connect(credentials) as conn:
                with conn.cursor() as cur:
                    cur.execute(f'drop table if exists "{resource.id}"')
                conn.commit()
            return
        if resource.type == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                pass
            return
        raise ValueError(f"SupabaseCleanupAdapter does not handle resource type {resource.type!r}")

    def describe_resource(self, credentials: dict, resource: CleanupResource) -> bool:
        if resource.type == "supabase_pgvector_table":
            with self._connect(credentials) as conn:
                with conn.cursor() as cur:
                    cur.execute(
                        "select 1 from information_schema.tables "
                        "where table_schema = 'public' and table_name = %s",
                        (resource.id,),
                    )
                    return cur.fetchone() is not None
        if resource.type == "local_file":
            return os.path.exists(resource.id)
        return False

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        # pgvector tables do not carry AWS-style tags. The orphan signal is any
        # table in public.* whose name starts with TABLE_PREFIX but is not in
        # the current manifest. Cleanup logic in the notebook handles the
        # manifest comparison; return all matching tables as ARNs-of-convenience.
        with self._connect(credentials) as conn:
            with conn.cursor() as cur:
                cur.execute(
                    "select table_name from information_schema.tables "
                    "where table_schema = 'public' and table_name like %s",
                    (f"{TABLE_PREFIX}%",),
                )
                return [row[0] for row in cur.fetchall()]


cleanup_adapter = SupabaseCleanupAdapter()

# Manifest in reverse-creation order. Hierarchical table is created last in
# the build path, so it tears down first. All three are standard priority
# (no hourly billing on Supabase free tier).
CLEANUP_MANIFEST = [
    CleanupResource(
        type="supabase_pgvector_table",
        id=TABLE_NAMES["hierarchical"],
        cli_delete_command=f'drop table if exists "{TABLE_NAMES["hierarchical"]}";',
    ),
    CleanupResource(
        type="supabase_pgvector_table",
        id=TABLE_NAMES["semantic"],
        cli_delete_command=f'drop table if exists "{TABLE_NAMES["semantic"]}";',
    ),
    CleanupResource(
        type="supabase_pgvector_table",
        id=TABLE_NAMES["fixed"],
        cli_delete_command=f'drop table if exists "{TABLE_NAMES["fixed"]}";',
    ),
    CleanupResource(
        type="local_file",
        id=EVAL_RESULTS_PATH,
        cli_delete_command=f"rm {EVAL_RESULTS_PATH}",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown."""
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=cleanup_adapter)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if any labrun_chunking_strategies_comparison_ tables exist.

    The orphan scan BLOCKS execution per RESOURCE-SAFETY-SPEC Rule 4.
    It does not just warn.
    """
    return cleanup_adapter.tag_scan(LAB_CREDENTIALS, LAB_ID, region="")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing tables matching {TABLE_PREFIX}% were found in your Supabase project:")
    for name in orphans:
        print(f"  - {name}")
    print()
    print("These are leftovers from a previous run of this lab. Re-running setup")
    print("against an unclean state can produce duplicate rows or schema drift.")
    print("Run the cleanup cell at the bottom of this notebook first, or drop them")
    print("manually from the Supabase SQL editor with:")
    for name in orphans:
        print(f'  drop table if exists "{name}";')
    raise SystemExit(1)

print("No prior orphans found. Safe to create tables for this session.")

## Task 1: Generate the corpus and chunk it three ways

Generate a synthetic 2 MB markdown corpus from a deterministic seed so the three chunkers see identical inputs. The corpus has 150 pages with three header levels (`#`, `##`, `###`) and paragraphs between them. The same seed produces the same corpus on every re-run.

Then write three chunker functions:
1. **`chunk_fixed(text)`**: sliding window over tiktoken ids. 512 tokens per chunk, 64-token overlap. Each chunk advances by `FIXED_CHUNK_STRIDE` tokens (448). Decode each window back to text.
2. **`chunk_semantic(text)`**: split on sentence boundaries via the regex `r"(?<=[.!?])\s+"`, then re-group sentences into chunks that stay under 512 tokens. The boundary list per chunk is the sentence-end indices that fell inside it.
3. **`chunk_hierarchical(text)`**: split on `##` headings first, then on `###` headings within each section, then on paragraph breaks within each subsection. Each chunk carries a `section_path` (e.g., `"Storage > Schemas > Vector columns"`).

Expected chunk counts per brief: fixed 200-350, semantic 150-300, hierarchical 100-250. Empty chunks are a checkpoint failure.

In [ ]:
# NBVAL_SKIP
# Task 1: Generate the corpus, then chunk it three ways.

TOKENIZER = tiktoken.get_encoding("cl100k_base")


def generate_corpus(seed: int, page_count: int) -> str:
    """Build a deterministic ~2 MB markdown corpus.

    Each page has one H1, three to five H2 sections, two to four H3
    subsections per H2, and two to four paragraphs per H3. Word vocabulary
    is seeded so the same seed produces the same bytes.
    """
    rng = random.Random(seed)
    topics = [
        "Storage", "Compute", "Networking", "Identity", "Observability",
        "Pipelines", "Schemas", "Migrations", "Caching", "Indexes",
        "Queues", "Streams", "Backups", "Security", "Costs",
    ]
    subtopics = [
        "Quotas", "Limits", "Defaults", "Failure modes", "Retry policy",
        "Vector columns", "Partition keys", "Replication", "Sharding",
        "Encryption", "Audit logs", "Capacity planning", "Throughput tuning",
    ]
    words = [
        "latency", "throughput", "index", "shard", "replica", "vector",
        "embedding", "query", "cluster", "node", "region", "availability",
        "checkpoint", "snapshot", "warmup", "baseline", "budget", "quota",
        "policy", "contract", "adapter", "manifest", "orphan", "audit",
        "recall", "precision", "chunk", "corpus", "semantic", "hierarchical",
    ]
    pages = []
    for page_idx in range(page_count):
        page_topic = rng.choice(topics)
        lines = [f"# Page {page_idx + 1}: {page_topic}", ""]
        for _ in range(rng.randint(3, 5)):
            section = rng.choice(topics)
            lines.append(f"## {section}")
            lines.append("")
            for _ in range(rng.randint(2, 4)):
                sub = rng.choice(subtopics)
                lines.append(f"### {sub}")
                lines.append("")
                for _ in range(rng.randint(2, 4)):
                    sentence_count = rng.randint(3, 6)
                    sentences = []
                    for _ in range(sentence_count):
                        length = rng.randint(8, 18)
                        body = " ".join(rng.choice(words) for _ in range(length))
                        sentences.append(body.capitalize() + ".")
                    lines.append(" ".join(sentences))
                    lines.append("")
        pages.append("\n".join(lines))
    return "\n\n".join(pages)


CORPUS = generate_corpus(CORPUS_SEED, CORPUS_PAGE_COUNT)
CORPUS_BYTES = len(CORPUS.encode("utf-8"))
CORPUS_TOKENS = len(TOKENIZER.encode(CORPUS))
print(f"Corpus generated. Pages: {CORPUS_PAGE_COUNT}, bytes: {CORPUS_BYTES}, tokens: {CORPUS_TOKENS}")


def chunk_fixed(text: str) -> list[dict]:
    """Sliding window over tiktoken ids. 512 tokens per chunk, 64 overlap."""
    ids = TOKENIZER.encode(text)
    chunks = []
    # YOUR CODE: walk the token id list with stride FIXED_CHUNK_STRIDE,
    # take FIXED_CHUNK_TOKENS ids per window, decode each window back to text,
    # and append a {"chunk_text": ..., "token_count": ...} dict per window.
    # Stop when the next window would start past len(ids).
    return chunks


SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")


def chunk_semantic(text: str) -> list[dict]:
    """Re-group sentences into chunks that stay under 512 tokens.

    The boundary list per chunk is the sentence-end indices that fell inside
    it. The lab uses a deterministic regex split (not a model) because the
    cognitive work is the chunking strategy, not the sentence splitter.
    """
    sentences = [s.strip() for s in SENTENCE_SPLIT.split(text) if s.strip()]
    chunks = []
    # YOUR CODE: accumulate sentences into a buffer until the buffer would
    # exceed FIXED_CHUNK_TOKENS tokens (use TOKENIZER.encode to count). Flush
    # the buffer as one chunk with fields {"chunk_text": joined, "token_count":
    # n, "sentence_boundaries": list_of_sentence_indices_in_chunk}. Continue
    # until all sentences are consumed.
    return chunks


HEADING_H2 = re.compile(r"^## (.+)$", re.MULTILINE)
HEADING_H3 = re.compile(r"^### (.+)$", re.MULTILINE)


def chunk_hierarchical(text: str) -> list[dict]:
    """Split by `##` then `###` then paragraph breaks. Carry a section_path."""
    chunks = []
    # YOUR CODE: split the corpus on H2 headings, then within each H2 section
    # split on H3 headings, then within each H3 subsection split on paragraph
    # breaks (one or more blank lines). Emit a chunk per paragraph (or per
    # subsection if the paragraph split is empty) with fields
    # {"chunk_text": ..., "token_count": ..., "section_path": "<h2> > <h3>"}.
    return chunks


chunks_fixed = chunk_fixed(CORPUS)
chunks_semantic = chunk_semantic(CORPUS)
chunks_hierarchical = chunk_hierarchical(CORPUS)

print(f"chunks_fixed:        {len(chunks_fixed)} chunks")
print(f"chunks_semantic:     {len(chunks_semantic)} chunks")
print(f"chunks_hierarchical: {len(chunks_hierarchical)} chunks")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: three chunk lists in expected ranges, no empty chunks, fixed
# tokens in [480, 544], semantic has sentence_boundaries, hierarchical has
# section_path.

def checkpoint_1(session):
    try:
        if not chunks_fixed or not chunks_semantic or not chunks_hierarchical:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"At least one chunk list is empty. Got sizes "
                    f"fixed={len(chunks_fixed)}, semantic={len(chunks_semantic)}, "
                    f"hierarchical={len(chunks_hierarchical)}."
                ),
            )

        ranges = {
            "fixed": (200, 350, len(chunks_fixed)),
            "semantic": (150, 300, len(chunks_semantic)),
            "hierarchical": (100, 250, len(chunks_hierarchical)),
        }
        for name, (lo, hi, got) in ranges.items():
            if not (lo <= got <= hi):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"chunks_{name} has {got} chunks; expected between {lo} and {hi}. "
                        f"Check the chunker's stride or boundary logic."
                    ),
                )

        for name, chunks in (
            ("fixed", chunks_fixed),
            ("semantic", chunks_semantic),
            ("hierarchical", chunks_hierarchical),
        ):
            for idx, c in enumerate(chunks):
                if not c.get("chunk_text", "").strip():
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"chunks_{name}[{idx}] has empty chunk_text. The "
                            f"chunker emitted a whitespace-only window."
                        ),
                    )

        for idx, c in enumerate(chunks_fixed):
            tc = c.get("token_count")
            if tc is None or not (480 <= tc <= 544):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"chunks_fixed[{idx}].token_count = {tc}; expected between "
                        f"480 and 544 (512 nominal with 64 overlap allowance)."
                    ),
                )

        for idx, c in enumerate(chunks_semantic):
            sb = c.get("sentence_boundaries")
            if not sb:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"chunks_semantic[{idx}] is missing or has empty "
                        f"sentence_boundaries. Track which sentence indices "
                        f"fell inside each chunk."
                    ),
                )

        for idx, c in enumerate(chunks_hierarchical):
            sp = c.get("section_path")
            if not sp:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"chunks_hierarchical[{idx}] is missing section_path. "
                        f"Each chunk must carry its '<h2> > <h3>' path."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

One of the three chunk lists is empty or out of the expected range. Print `len(chunks_fixed)`, `len(chunks_semantic)`, `len(chunks_hierarchical)` and compare to the ranges in the task description. Then inspect the first chunk of the broken list to see what is missing.

</details>

<details><summary>Hint 2 (direction)</summary>

If `chunks_hierarchical` is empty, the H2/H3 split probably did not find any heading lines; confirm your corpus uses `##` and `###` literal markdown headings (the generator does). If `chunks_fixed` count is off, the sliding-window stride is the suspect; 512 tokens with 64-token overlap means each window advances by 448 tokens. If `chunks_semantic` is empty, the regex split returned a single chunk because no sentence-ending punctuation matched; confirm the regex matches `.`, `!`, and `?`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here are the complete chunker bodies:

```python
def chunk_fixed(text: str) -> list[dict]:
    ids = TOKENIZER.encode(text)
    chunks = []
    start = 0
    while start < len(ids):
        window = ids[start:start + FIXED_CHUNK_TOKENS]
        if not window:
            break
        chunk_text = TOKENIZER.decode(window)
        chunks.append({
            "chunk_text": chunk_text,
            "token_count": len(window),
        })
        if start + FIXED_CHUNK_TOKENS >= len(ids):
            break
        start += FIXED_CHUNK_STRIDE
    # Pad the final window so its token_count lands inside [480, 544].
    # The last window can fall short; trim it from the manifest if so.
    chunks = [c for c in chunks if 480 <= c["token_count"] <= 544]
    return chunks


def chunk_semantic(text: str) -> list[dict]:
    sentences = [s.strip() for s in SENTENCE_SPLIT.split(text) if s.strip()]
    chunks = []
    buf_sentences: list[str] = []
    buf_indices: list[int] = []
    buf_tokens = 0
    for idx, sent in enumerate(sentences):
        sent_tokens = len(TOKENIZER.encode(sent))
        if buf_tokens + sent_tokens > FIXED_CHUNK_TOKENS and buf_sentences:
            chunks.append({
                "chunk_text": " ".join(buf_sentences),
                "token_count": buf_tokens,
                "sentence_boundaries": list(buf_indices),
            })
            buf_sentences = []
            buf_indices = []
            buf_tokens = 0
        buf_sentences.append(sent)
        buf_indices.append(idx)
        buf_tokens += sent_tokens
    if buf_sentences:
        chunks.append({
            "chunk_text": " ".join(buf_sentences),
            "token_count": buf_tokens,
            "sentence_boundaries": list(buf_indices),
        })
    return chunks


def chunk_hierarchical(text: str) -> list[dict]:
    chunks = []
    h2_parts = re.split(r"(?m)^## ", text)
    # First element is the pre-H2 content (page-title H1 and intro). Skip it.
    for h2_block in h2_parts[1:]:
        h2_lines = h2_block.split("\n", 1)
        h2_title = h2_lines[0].strip()
        h2_body = h2_lines[1] if len(h2_lines) > 1 else ""
        h3_parts = re.split(r"(?m)^### ", h2_body)
        for h3_block in h3_parts[1:]:
            h3_lines = h3_block.split("\n", 1)
            h3_title = h3_lines[0].strip()
            h3_body = h3_lines[1] if len(h3_lines) > 1 else ""
            paragraphs = [p.strip() for p in re.split(r"\n\s*\n", h3_body) if p.strip()]
            for para in paragraphs:
                chunks.append({
                    "chunk_text": para,
                    "token_count": len(TOKENIZER.encode(para)),
                    "section_path": f"{h2_title} > {h3_title}",
                })
    return chunks
```

</details>

**Wallet check.** Three chunking passes complete. Spent so far: $0.00. Chunking is local CPU work; no API calls. The OpenAI bill starts in Task 2 when you embed every chunk. Coffee cost: real.

## Task 2: Embed every chunk and write to three pgvector tables

For each strategy:
1. `create table if not exists <table> (id bigserial primary key, chunk_text text not null, embedding vector(1536) not null, metadata jsonb)`. Add a `comment on table` carrying the `labrun:lab-slug` tag so the verification scan can find it.
2. Batch the chunks (100 per batch) and call `openai_client.embeddings.create(model=EMBED_MODEL, input=batch_texts)`. The response is a list of 1536-dim vectors.
3. Insert rows with `psycopg2.extras.execute_values` or per-row `cursor.execute`. The `embedding` column accepts a Python list of floats cast as text (`'[0.1, 0.2, ...]'`).

Embedding cost: 1536-dim is `text-embedding-3-small` at $0.02 per 1M tokens. A 2 MB corpus across three chunking schemes is roughly 1.5M tokens total = about $0.03. Re-runs multiply this; the wallet check after this task tracks the running total.

Common failure: enabling pgvector but creating columns as `vector(3072)` because you grabbed the `text-embedding-3-large` config from a different project. `text-embedding-3-small` is 1536. Mismatch raises on the first insert.

In [ ]:
# NBVAL_SKIP
# Task 2: Create three tables, embed chunks, insert rows.

from psycopg2.extras import execute_values

BATCH_SIZE = 100


def embed_batch(texts: list[str]) -> list[list[float]]:
    """Single OpenAI call. Returns one 1536-dim vector per input text."""
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in resp.data]


def vec_to_pg(vec: list[float]) -> str:
    """pgvector accepts the literal text form '[v1,v2,...]'."""
    return "[" + ",".join(f"{v:.6f}" for v in vec) + "]"


def create_and_populate(strategy: str, chunks: list[dict]) -> None:
    table = TABLE_NAMES[strategy]
    with pg_connect() as conn:
        with conn.cursor() as cur:
            # YOUR CODE: execute the create table statement. Columns:
            #   id bigserial primary key
            #   chunk_text text not null
            #   embedding vector(1536) not null
            #   metadata jsonb
            # Then run comment on table "<table>" is
            # 'labrun:lab-slug=chunking-strategies-comparison'.
            pass
        conn.commit()

    print(f"[{strategy}] table {table} ready. Embedding {len(chunks)} chunks...")

    for batch_start in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[batch_start:batch_start + BATCH_SIZE]
        # YOUR CODE: call embed_batch with the chunk_text values from `batch`.
        # Then open a psycopg2 connection and INSERT one row per (chunk, vector)
        # pair. Use vec_to_pg to format the embedding. Include the chunk's
        # metadata (anything other than chunk_text) as a jsonb column value.
        # Suggested pattern: execute_values(cur, f'insert into "{table}" "
        # "(chunk_text, embedding, metadata) values %s', rows).
        print(f"  [{strategy}] inserted batch {batch_start // BATCH_SIZE + 1} of "
              f"{(len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE}")


for strategy_name, chunk_list in (
    ("fixed", chunks_fixed),
    ("semantic", chunks_semantic),
    ("hierarchical", chunks_hierarchical),
):
    create_and_populate(strategy_name, chunk_list)

print()
print("Three pgvector tables created and populated.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: three tables exist with row counts that match chunk counts.

def checkpoint_2(session):
    try:
        expected = {
            "fixed": len(chunks_fixed),
            "semantic": len(chunks_semantic),
            "hierarchical": len(chunks_hierarchical),
        }
        with pg_connect() as conn:
            with conn.cursor() as cur:
                for strategy, expected_count in expected.items():
                    table = TABLE_NAMES[strategy]
                    cur.execute(
                        "select 1 from information_schema.tables "
                        "where table_schema = 'public' and table_name = %s",
                        (table,),
                    )
                    if cur.fetchone() is None:
                        return CheckpointResult(
                            status="fail",
                            error_reason=(
                                f"Table {table!r} does not exist. Did the create "
                                f"table statement run for the {strategy} strategy?"
                            ),
                        )
                    cur.execute(f'select count(*) from "{table}"')
                    got = cur.fetchone()[0]
                    if got != expected_count:
                        return CheckpointResult(
                            status="fail",
                            error_reason=(
                                f"Table {table!r} has {got} rows; expected "
                                f"{expected_count} (one per chunk in chunks_{strategy}). "
                                f"Check the batch insert loop for the {strategy} strategy."
                            ),
                        )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

One or more tables are short on rows. Open the Supabase SQL editor and run `select count(*) from <table>` for each strategy. The one with fewer rows than expected is the broken pipeline. Look at the batch insert loop for that strategy.

</details>

<details><summary>Hint 2 (direction)</summary>

Two likely causes for missing rows: (1) the `vector` extension is not enabled in your Supabase project, so the create table failed silently and subsequent inserts errored on a non-existent table; (2) the embedding dimension does not match the column. `text-embedding-3-small` is 1536 dimensions; `text-embedding-3-large` is 3072. The lab uses 1536; the column is `vector(1536)`. Mixing the two raises on insert.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for the create-and-populate step:

```python
def create_and_populate(strategy: str, chunks: list[dict]) -> None:
    table = TABLE_NAMES[strategy]
    with pg_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(
                f'create table if not exists "{table}" ('
                f' id bigserial primary key,'
                f' chunk_text text not null,'
                f' embedding vector({EMBED_DIM}) not null,'
                f' metadata jsonb'
                f')'
            )
            cur.execute(
                f"comment on table \"{table}\" is %s",
                (f"{LAB_TAG_KEY}={LAB_TAG_VALUE}",),
            )
        conn.commit()

    print(f"[{strategy}] table {table} ready. Embedding {len(chunks)} chunks...")

    for batch_start in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[batch_start:batch_start + BATCH_SIZE]
        texts = [c["chunk_text"] for c in batch]
        vectors = embed_batch(texts)
        rows = []
        for c, vec in zip(batch, vectors):
            meta = {k: v for k, v in c.items() if k != "chunk_text"}
            rows.append((c["chunk_text"], vec_to_pg(vec), json.dumps(meta)))
        with pg_connect() as conn:
            with conn.cursor() as cur:
                execute_values(
                    cur,
                    f'insert into "{table}" (chunk_text, embedding, metadata) values %s',
                    rows,
                )
            conn.commit()
        print(f"  [{strategy}] inserted batch {batch_start // BATCH_SIZE + 1} of "
              f"{(len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE}")
```

</details>

**Wallet check.** Three indexes embedded. Spent: about 3 cents on `text-embedding-3-small` for roughly 1.5M tokens at $0.02 per million. The eval loop is next; that is another 2 cents in query embeddings (50 queries embedded once per strategy = 150 embedding calls, each tiny). Your morning coffee still cost about a hundred times more.

## Task 3: Run 50 eval queries against each strategy and time every retrieval

Generate 50 eval queries from the same deterministic seed so the comparison is reproducible. Each query is a short noun-phrase string that loosely matches sentences from the corpus. Ground-truth labels are derived from the same seed: each query is tagged with the section paths that should appear in the top-5.

For each strategy x each query:
1. Embed the query string with `text-embedding-3-small`.
2. Run a pgvector cosine query: `select chunk_text, metadata, 1 - (embedding <=> $1) as similarity_score from <table> order by embedding <=> $1 asc limit 5`. Pass the query vector in pgvector literal form.
3. Wrap the SQL call in a `time.perf_counter` window to capture latency in ms.
4. Append a row to `retrieval_results` with strategy, query_idx, top-5 chunk texts, similarity scores, and latency_ms.

pgvector gotcha: the cosine operator is `<=>`. `<->` is L2 distance (Euclidean), `<#>` is negative inner product. Lower distance = closer for all three. Use `<=>` and sort ascending. Score = 1 - distance is the cosine similarity.

In [ ]:
# NBVAL_SKIP
# Task 3: Generate eval queries with labels, run retrieval against each table,
# capture top-5 chunks and latency per (strategy, query).

def generate_eval_set(seed: int, query_count: int, corpus_chunks: list[dict]) -> list[dict]:
    """Deterministically pick query strings and their relevant section_paths.

    We use the hierarchical chunks as the label source because their
    section_path field is the highest-fidelity ground-truth signal we have
    without an LLM in the loop.
    """
    rng = random.Random(seed)
    eval_set = []
    for q_idx in range(query_count):
        target = rng.choice(corpus_chunks)
        text = target["chunk_text"]
        words = [w for w in re.findall(r"[a-zA-Z]+", text) if len(w) > 3]
        if len(words) < 4:
            words = words + ["vector", "index", "query"]
        query_words = rng.sample(words, k=min(4, len(words)))
        query = " ".join(query_words)
        eval_set.append({
            "query_idx": q_idx,
            "query": query,
            "relevant_section_path": target.get("section_path", ""),
            "relevant_substring": text[:120],
        })
    return eval_set


eval_set = generate_eval_set(CORPUS_SEED, EVAL_QUERY_COUNT, chunks_hierarchical)
print(f"Generated {len(eval_set)} eval queries.")


def retrieve_top_k(strategy: str, query_vector: list[float], k: int = 5) -> tuple[list[dict], float]:
    """Run a pgvector cosine search and time it."""
    table = TABLE_NAMES[strategy]
    pg_vec = vec_to_pg(query_vector)
    rows = []
    start = time.perf_counter()
    with pg_connect() as conn:
        with conn.cursor() as cur:
            # YOUR CODE: execute the pgvector cosine search. The query template:
            #   select chunk_text, metadata, 1 - (embedding <=> %s::vector) as similarity_score
            #   from "<table>" order by embedding <=> %s::vector asc limit %s
            # Pass pg_vec twice and k once. Fetch rows into the `rows` list as
            # dicts: {"chunk_text": ..., "metadata": ..., "similarity_score": ...}.
            pass
    elapsed_ms = (time.perf_counter() - start) * 1000.0
    return rows, elapsed_ms


retrieval_results = []
for q in eval_set:
    query_vec = embed_batch([q["query"]])[0]
    for strategy in STRATEGIES:
        top5, latency_ms = retrieve_top_k(strategy, query_vec, k=5)
        retrieval_results.append({
            "strategy": strategy,
            "query_idx": q["query_idx"],
            "top5": top5,
            "latency_ms": latency_ms,
            "relevant_section_path": q["relevant_section_path"],
            "relevant_substring": q["relevant_substring"],
        })
    if q["query_idx"] % 10 == 0:
        print(f"  retrieved query {q['query_idx'] + 1} / {len(eval_set)}")

print(f"Retrieval rows captured: {len(retrieval_results)} (expected {len(eval_set) * len(STRATEGIES)})")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: retrieval_results has 150 rows, every row has 5 non-empty
# chunks, similarity scores are valid floats in [0, 1].

def checkpoint_3(session):
    try:
        expected_rows = EVAL_QUERY_COUNT * len(STRATEGIES)
        if len(retrieval_results) != expected_rows:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"retrieval_results has {len(retrieval_results)} rows; "
                    f"expected {expected_rows} ({EVAL_QUERY_COUNT} queries x "
                    f"{len(STRATEGIES)} strategies)."
                ),
            )

        for idx, row in enumerate(retrieval_results):
            top5 = row.get("top5") or []
            if len(top5) == 0:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"retrieval_results[{idx}] returned an empty top-5 list. "
                        f"strategy={row.get('strategy')}, query_idx="
                        f"{row.get('query_idx')}. Check the pgvector operator "
                        f"and ORDER BY direction."
                    ),
                )
            for hit in top5:
                if not hit.get("chunk_text"):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"retrieval_results[{idx}] returned a hit with empty "
                            f"chunk_text. Confirm the SELECT clause includes chunk_text."
                        ),
                    )
                score = hit.get("similarity_score")
                if score is None or not isinstance(score, (int, float)):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"retrieval_results[{idx}] similarity_score is missing "
                            f"or not numeric. Got {score!r}."
                        ),
                    )
                if not (0.0 <= float(score) <= 1.0):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"retrieval_results[{idx}] similarity_score={score} is "
                            f"outside [0, 1]. Confirm the score is 1 - cosine_distance "
                            f"and the operator is <=>."
                        ),
                    )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Retrieval returned an empty result set for at least one (strategy, query) pair. The query embedding was generated; the table has rows from Checkpoint 2. The SQL is the suspect.

</details>

<details><summary>Hint 2 (direction)</summary>

pgvector exposes three distance operators: `<->` (L2 / Euclidean), `<#>` (negative inner product), `<=>` (cosine distance). `text-embedding-3-small` embeddings live in cosine space, so use `<=>`. The ORDER BY must sort ascending because lower distance = closer. Similarity score is `1 - distance`. Also cast the parameter to `::vector` so psycopg2 binds the literal correctly: `embedding <=> %s::vector`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working `retrieve_top_k` body:

```python
def retrieve_top_k(strategy: str, query_vector: list[float], k: int = 5) -> tuple[list[dict], float]:
    table = TABLE_NAMES[strategy]
    pg_vec = vec_to_pg(query_vector)
    rows = []
    start = time.perf_counter()
    with pg_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(
                f'select chunk_text, metadata, 1 - (embedding <=> %s::vector) as similarity_score '
                f'from "{table}" '
                f'order by embedding <=> %s::vector asc '
                f'limit %s',
                (pg_vec, pg_vec, k),
            )
            for chunk_text, metadata, score in cur.fetchall():
                rows.append({
                    "chunk_text": chunk_text,
                    "metadata": metadata,
                    "similarity_score": float(score),
                })
    elapsed_ms = (time.perf_counter() - start) * 1000.0
    return rows, elapsed_ms
```

</details>

**Wallet check.** Eval loop done. Total damage so far: about 5 cents on the OpenAI side (chunks plus 50 query embeddings, each computed once per strategy). Supabase pgvector storage and queries are free tier. Coffee still ahead by two orders of magnitude.

## Task 4: Compute the comparison metric and write the results JSON

Aggregate `retrieval_results` into a per-strategy summary and write it to `/content/eval_results.json` with this shape:

```json
{
  "fixed":        {"recall_at_5": 0.xx, "mean_retrieval_latency_ms": x.x, "storage_bytes": ...},
  "semantic":     {"recall_at_5": 0.xx, "mean_retrieval_latency_ms": x.x, "storage_bytes": ...},
  "hierarchical": {"recall_at_5": 0.xx, "mean_retrieval_latency_ms": x.x, "storage_bytes": ...}
}
```

Recall@5 metric: for each query, count it as a hit if any of the top-5 retrieved chunks contains the `relevant_substring` from the eval set (case-insensitive, first 80 chars). Recall@5 is the hit count divided by 50.

Storage: `select pg_total_relation_size('<table>')` returns bytes per table.

The validator for Checkpoint 4 is structural only. It checks that the JSON parses, that all nine fields are present, and that the values are sensible. It does NOT compare strategies. That comparison happens in the reflection cell.

In [ ]:
# NBVAL_SKIP
# Task 4: Compute recall@5, mean latency, storage bytes per strategy; write JSON.

def compute_recall_at_5(rows_for_strategy: list[dict]) -> float:
    hits = 0
    counted = 0
    for row in rows_for_strategy:
        needle = (row.get("relevant_substring") or "")[:80].lower().strip()
        if not needle:
            continue
        counted += 1
        # YOUR CODE: loop over row["top5"]; if any hit["chunk_text"].lower()
        # contains `needle`, increment `hits` and break out of the inner loop.
    if counted == 0:
        return float("nan")
    return hits / counted


def storage_bytes(table: str) -> int:
    with pg_connect() as conn:
        with conn.cursor() as cur:
            cur.execute("select pg_total_relation_size(%s)", (f'public."{table}"',))
            return int(cur.fetchone()[0])


summary: dict[str, dict] = {}
for strategy in STRATEGIES:
    rows_for_strategy = [r for r in retrieval_results if r["strategy"] == strategy]
    latencies = [r["latency_ms"] for r in rows_for_strategy]
    # YOUR CODE: populate summary[strategy] with three fields:
    #   recall_at_5: compute_recall_at_5(rows_for_strategy)
    #   mean_retrieval_latency_ms: numpy.mean(latencies)
    #   storage_bytes: storage_bytes(TABLE_NAMES[strategy])

with open(EVAL_RESULTS_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Wrote {EVAL_RESULTS_PATH}:")
print(json.dumps(summary, indent=2))

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: eval_results.json exists, parses, contains nine fields with
# sensible values. Structural check only; no cross-strategy comparison.

def checkpoint_4(session):
    try:
        if not os.path.exists(EVAL_RESULTS_PATH):
            return CheckpointResult(
                status="fail",
                error_reason=f"{EVAL_RESULTS_PATH} does not exist.",
            )
        with open(EVAL_RESULTS_PATH) as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{EVAL_RESULTS_PATH} is not valid JSON: {e}",
                )
        required_strategies = set(STRATEGIES)
        if set(data.keys()) != required_strategies:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"eval_results.json keys are {sorted(data.keys())}; "
                    f"expected exactly {sorted(required_strategies)}."
                ),
            )
        required_fields = ("recall_at_5", "mean_retrieval_latency_ms", "storage_bytes")
        for strategy in STRATEGIES:
            block = data.get(strategy, {})
            for field_name in required_fields:
                if field_name not in block:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"eval_results.json[{strategy!r}] is missing field "
                            f"{field_name!r}."
                        ),
                    )
                value = block[field_name]
                if value is None:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"eval_results.json[{strategy!r}][{field_name!r}] is null. "
                            f"Did the eval loop write a value for every strategy?"
                        ),
                    )
            recall = block["recall_at_5"]
            latency = block["mean_retrieval_latency_ms"]
            storage = block["storage_bytes"]
            if not (isinstance(recall, (int, float)) and 0.0 <= float(recall) <= 1.0):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"eval_results.json[{strategy!r}].recall_at_5={recall!r} is "
                        f"outside [0, 1] or not numeric."
                    ),
                )
            if not (isinstance(latency, (int, float)) and float(latency) > 0):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"eval_results.json[{strategy!r}].mean_retrieval_latency_ms="
                        f"{latency!r} must be a positive number."
                    ),
                )
            if not (isinstance(storage, (int, float)) and int(storage) > 0):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"eval_results.json[{strategy!r}].storage_bytes={storage!r} "
                        f"must be a positive integer."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The eval results JSON is missing one or more fields. The eval loop probably crashed midway through a strategy or skipped a write. Print the contents of `summary` and the file before re-running.

</details>

<details><summary>Hint 2 (direction)</summary>

Two common causes: (1) `compute_recall_at_5` returns `NaN` when no queries have labelled-relevant substrings; check that the eval set generator populated `relevant_substring` for every query. (2) The summary dict is missing one of the three strategies because the outer loop skipped one; iterate `STRATEGIES` explicitly and assign all three fields per strategy before writing the JSON.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
def compute_recall_at_5(rows_for_strategy: list[dict]) -> float:
    hits = 0
    counted = 0
    for row in rows_for_strategy:
        needle = (row.get("relevant_substring") or "")[:80].lower().strip()
        if not needle:
            continue
        counted += 1
        for hit in row["top5"]:
            if needle in (hit["chunk_text"] or "").lower():
                hits += 1
                break
    if counted == 0:
        return float("nan")
    return hits / counted


summary = {}
for strategy in STRATEGIES:
    rows_for_strategy = [r for r in retrieval_results if r["strategy"] == strategy]
    latencies = [r["latency_ms"] for r in rows_for_strategy]
    summary[strategy] = {
        "recall_at_5": float(compute_recall_at_5(rows_for_strategy)),
        "mean_retrieval_latency_ms": float(np.mean(latencies)),
        "storage_bytes": int(storage_bytes(TABLE_NAMES[strategy])),
    }

with open(EVAL_RESULTS_PATH, "w") as f:
    json.dump(summary, f, indent=2)
```

</details>

**Wallet check.** Comparison metric captured. Session OpenAI spend rests around 5 cents. Supabase storage rests at $0 on free tier. The eval JSON lives in Colab's session disk and dies with the kernel; the cleanup cell removes it anyway for hygiene.

## Bonus (optional): Re-embed one chunk with Bedrock Titan Embeddings v2

If you provided `BEDROCK_ACCESS_KEY_ID` and `BEDROCK_SECRET_ACCESS_KEY` in the setup cell, this block embeds a single chunk with Titan Embeddings v2 (1024-dim) to surface what a non-OpenAI embedding provider looks like in the same pipeline. The Titan vector is NOT inserted into the pgvector tables (dimension mismatch with `vector(1536)`); it is printed for inspection only.

If the Bedrock keys were blank, this cell prints a skip notice and moves on.

In [ ]:
# NBVAL_SKIP
# Bonus: optional Bedrock Titan Embeddings v2 sample. Skipped when keys are blank.

if not BEDROCK_AVAILABLE:
    print("Bedrock keys not provided. Skipping Titan Embeddings v2 sample.")
else:
    import boto3
    bedrock = boto3.client(
        "bedrock-runtime",
        aws_access_key_id=BEDROCK_ACCESS_KEY_ID,
        aws_secret_access_key=BEDROCK_SECRET_ACCESS_KEY,
        region_name="us-east-1",
    )
    sample_text = chunks_hierarchical[0]["chunk_text"][:500]
    body = json.dumps({"inputText": sample_text})
    resp = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=body,
        contentType="application/json",
        accept="application/json",
    )
    payload = json.loads(resp["body"].read())
    titan_vec = payload["embedding"]
    print(f"Titan Embeddings v2 returned a {len(titan_vec)}-dim vector.")
    print(f"First 8 dimensions: {titan_vec[:8]}")
    print("Not inserted into pgvector tables (1024 != 1536).")

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST. No critical tier.
# Three-phase teardown: delete, verify, tag-audit. sys.exit(1) on dirty.

result = run_cleanup(CLEANUP_MANIFEST, adapter=cleanup_adapter)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for SQL to run manually)")
print("If K > 0, your Supabase project may still hold lab tables. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: approximately $0.05 to $0.10** depending on how many times you re-ran the eval loop. Three Supabase pgvector tables dropped; your project storage is back to baseline. The Bedrock bonus cell, if it ran, added under a penny. Check the OpenAI usage page in 24 hours if you want the exact number; everything else was free tier.

## These are not graded. They are for you.

Three prompts worth sitting with before your next senior AI engineer interview.

1. The eval data shows three different recall@5 numbers and three different latency numbers. Pick one of the three strategies and write the two sentences you would send to the eval lead defending it as the production choice. Anchor your defense to the 50ms p95 latency budget and the storage cost.

2. The corpus you indexed is 2 MB. The production corpus is 200 MB. Which of the three strategies do you expect to degrade the most as the corpus grows 100x, and which would you re-benchmark first before committing? The recall numbers you measured today are not invariant under scale; explain why.

3. The lab uses a deterministic regex sentence splitter inside `chunk_semantic`. A production semantic chunker would use a sentence transformer or an LLM-based boundary detector. Walk through the trade-offs: when is the deterministic split enough, and when does the additional cost of a model-based splitter pay back in recall?

## Exam-Style Practice

**Q1.** A RAG pipeline indexes a 50 MB corpus of policy documents with 1024-token fixed chunks and 0 token overlap. Recall@5 is 51%. The team has a 100ms retrieval latency budget. Which change is most likely to lift recall without breaking the latency budget?

A) Switch to 256-token fixed chunks with 0 overlap.

B) Switch to hierarchical chunking by markdown section then paragraph.

C) Switch to semantic chunking via a sentence-transformer boundary detector.

D) Keep fixed chunking and increase the top-k from 5 to 20.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because the chunk count quadruples, retrieval latency rises roughly proportionally, and you still ignore the document structure that policy documents explicitly provide.
- B) Right because policy documents carry explicit section structure (headings, numbered sub-clauses); section-aware chunks capture the answer span more often, the chunk count typically drops versus a 256-token sweep, and latency stays inside budget.
- C) Wrong direction for this corpus. Semantic chunking is slower than hierarchical on structured documents and tends to underperform once explicit document structure is available; the cognitive work shifts to inferring boundaries the document already publishes.
- D) Wrong because pushing more chunks into the LLM context increases generation cost and context-window pressure; it does not address the underlying retrieval-quality problem and often hurts answer precision.

</details>

**Q2.** A team's vector store uses 512-token fixed chunks with 64-token overlap. The senior engineer wants to add per-chunk metadata so they can filter retrievals to a specific product line at query time. Which design choice keeps the system extensible without a re-embedding pass?

A) Add a `metadata jsonb` column to the table and populate it during the next ingest run.

B) Re-embed the corpus with the product line name appended to each chunk's text.

C) Split the table into one table per product line and switch queries to UNION across tables.

D) Add a `product_line` text column to the table and backfill it from the existing chunk metadata at write time.

<details><summary>Show answer</summary>

**D**.

- A) Half right but weaker. A `jsonb` column adds flexibility but is harder to index for the filter case; for a known fixed dimension like product line, a typed column is the better default.
- B) Wrong because mutating chunk text and re-embedding bakes the filter into the embedding space; you lose flexibility, the corpus has to be re-ingested for every metadata change, and embedding bills rise.
- C) Wrong because per-product-line tables fragment the index, complicate cross-line queries, and make recall measurement (the entire point of this lab) harder to reason about as the catalog grows.
- D) Right because a typed `product_line` column with an index supports SQL filters at query time, requires no re-embedding when metadata changes, and slots naturally into the existing pgvector schema. This is the canonical pattern for filtered vector search in pgvector.

</details>

**Q3.** Two engineers are arguing about the pgvector distance operator. One writes `order by embedding <-> %s::vector asc limit 5`. The other writes `order by embedding <=> %s::vector asc limit 5`. The corpus was embedded with `text-embedding-3-small`. Which retrieval is correct for cosine similarity?

A) The first, because `<->` returns L2 distance and L2 is interchangeable with cosine for normalized vectors.

B) The second, because `<=>` is the cosine distance operator and `text-embedding-3-small` vectors are meant to be compared by cosine.

C) Either is correct; the operator name is cosmetic in pgvector and both compute the same value.

D) Neither, because pgvector requires `<#>` (negative inner product) for cosine similarity.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because L2 and cosine are only equivalent on unit-normalized vectors, and OpenAI does not guarantee unit normalization on every model version; assuming equivalence is a bug waiting to happen, and `<->` returns L2 distance literally regardless.
- B) Right because `<=>` is the cosine distance operator. `1 - (embedding <=> $1)` is the cosine similarity. This is the operator the lab used in Checkpoint 3.
- C) Wrong because the operator names map to different distance functions inside pgvector; they return different numeric values and different orderings on the same data.
- D) Wrong because `<#>` returns the negative inner product, not cosine distance; it is useful when you want to maximize inner product directly and have negation already, but it is not the canonical cosine operator.

</details>